Compare active and idle states of IOT devices

Code structure: if possible have data and display independent, so that time can be saved by not recomputing unchanged data

e.g 
```python
show_data(get_data())
```

rather than
```python
def show_data():
    get_data()
```

also increases configurability

Terms:
* Class
* Window (w seconds long)
* Shift (s seconds)

In [ ]:
# function defs

THIRTY_MINUTES = 1750  # approx number of samples in 30 minutes

from enum import StrEnum
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pandas as pd
import numpy as np
import os
from colorhash import ColorHash
from functools import reduce
from collections import Counter
import matplotlib.style as mplstyle
from dataclasses import dataclass

mplstyle.use("fast")
import doctest


class State(StrEnum):
    ACTIVE = "active"
    IDLE = "idle"


class Data:
    def __init__(self, context, device: Device, state: State):
        self.device = device
        self.state = state
        data = pd.read_csv(
            f"{context.path}/{device.category}/{device}/{state}.csv",
            parse_dates=["timestamp"],
        )

        if device.name == "nestmini2":
            data["power_mW"] = data["power_W"]

        split = THIRTY_MINUTES
        self.training = data.iloc[:split, :]
        self.testing = data.iloc[split:, :]


class Context:
    """
    Get all the devices and categories.

    Hard coded to remove echodot3 since the idle data is truncated and can't be compared

    >>> len(Context('../PowerIDS/classifier/power_csv').devices)
    32

    >>> Context('../PowerIDS/classifier/power_csv').categories
    ['appliance', 'audio', 'camera', 'lights', 'tv']

    >>> len(Context('../PowerIDS/classifier/power_csv').data)
    64
    """

    def __init__(self, path: str):
        self.path = path
        self.categories = sorted(os.listdir(path))
        self.devices = [
            Device(device, category)
            for category in self.categories
            for device in os.listdir(f"{path}/{category}")
            if device != "echodot3"
        ]
        # The global pool of data for all the states we care about
        self.data = [
            Data(self, device, state) for state in State for device in self.devices
        ]


class Device:
    def __init__(self, name: str, category):
        self.name = name
        self.category = category
        # TODO nice names for devices, just a lookup table

    def __str__(self):
        return self.name


class Experiment:
    """
    Verify the new devices are seperate
    >>> e = Context('../PowerIDS/classifier/power_csv')
    >>> exp = Experiment(e, 180, 10, State.IDLE)
    >>> e.devices.pop() and len(exp.devices)
    32

    >>> e = Context('../PowerIDS/classifier/power_csv')
    >>> len(Experiment(e, 180, 10, State.IDLE).data)
    32
    """

    context: Context
    devices: list
    w: int
    s: int
    state: State

    def __init__(self, context: Context, w, s, state: State, include=[]):
        self.context = context
        if include == []:
            include = list(map(lambda device: device.name, context.devices))
        self.devices = [device for device in context.devices if device.name in include]
        self.data = [
            data for data in context.data if (data.device.name in include and data.state == state)
        ]  # if data.device.name in include and data.state == state]
        self.w = w
        self.s = s
        self.state = state


def sort_dict_by_value(dict, key=lambda x: x[1], reverse=False):
    """
    >>> sort_dict_by_value({'a': 2, 'b': 3, 'c': 1})
    {'c': 1, 'a': 2, 'b': 3}
    """

    return {k: v for k, v in sorted(dict.items(), key=key, reverse=reverse)}


# def window(df, samples, offset):
#     return df.iloc[offset: offset + samples]

# def intervalise(metric):
#     return pd.Interval(min(metric), max(metric))

# def compute_profile(df, s, o):
#     """
#     run all windows, return metrics in the form {'metric': [value]}, where the values are sorted

#     the first window takes up s samples, so there are len(data) - s samples left. each addional window uses up o spaces.

# >>> len(profile(pd.DataFrame(data={'power_mW': [1] * 100}), 10, 2)['mean'])
# 46

#     Additonal partial windows aren't counted
# >>> len(profile(pd.DataFrame(data={'power_mW': [1] * 101}), 10, 2)['mean'])
# 46

#   Non overlapping windows
# >>> len(profile(pd.DataFrame(data={'power_mW': [1] * 100}), 10, 20)['mean'])
# 5

# >>> len(profile(pd.DataFrame(data={'power_mW': [1] * 1750}), 400, 10)['mean'])
# 136
#    """
#     means = []
#     stds = []

#     # TODO do we want non rectangular windows? are edge effects important

#     for bin in range((len(df) - s) // o + 1):
#         window_power = window(df, s, o * bin).power_mW
#         means.append(window_power.mean())
#         stds.append(window_power.std())

#     return {'mean': sorted(means), 'std': sorted(stds)}

# """
# Assuming sorted lists, return the number of non overlapping intervals

# >>> overlap(([1, 3], [2, 4], [10, 20]))
# 1
# """
# def overlap(intervals, metrics):
#     # TODO: look at multiple metrics here
#     independents = 0
#     for i, interval in enumerate(intervals):
#         independent = True
#         for j, compare in enumerate(intervals):
#             if i != j:
#                 overlaps = 0
#                 for metric in metrics:
#                     if interval[metric][0] <= compare[metric][-1] and compare[metric][0] <= interval[metric][-1]:
#                         overlaps += 1
#                 if overlaps == len(metrics):
#                     independent = False
#         if independent:
#             independents += 1

#     return independents


# def compute_overlap(data, samples, offset, metrics=['mean']):
#     results = {device: compute_profile(df, samples, offset) for device, (df, _) in data.items()}
#     return overlap(results.values(), metrics)


def apply_windows(exp):
    """
    apply windows to data
    30 minutes training, 10 minutes testing

    return the data, graph it seperately
    """
    # just get the profiles as a list
    results = {
        device: compute_profile(training, samples, offset)
        for device, (training, _) in exp.data.items()
    }
    # assign the pr
    matches = {}
    for device, (_, testing) in exp.data.items():
        matches[device] = {}
        for metric in metrics:
            matches[device][metric] = []
            for point in compute_profile(testing, samples, offset)[metric]:
                # TODO fix the double computation here
                matches[device][metric].append(
                    reduce(
                        lambda old, new: (
                            new
                            if abs(np.median(new[1][metric]) - point)
                            < abs(np.median(old[1][metric]) - point)
                            else old
                        ),
                        results.items(),
                    )[0]
                )
    return matches


#  #| label: colourmesh

# def my_colormesh(fig, ax, x, y, z):
#     cmap = "BuPu"
#     im = ax.pcolormesh(x, y, z, cmap=cmap, shading='nearest', edgecolors='white', linewidths=0.1)
#     ax.tick_params('x', rotation=90)

#     # colorizer = mpl.colorizer.Colorizer(norm=mpl.colors.Normalize(0, 1), cmap=cmap)
#     # mpl.colorizer.ColorizingArtist(colorizer),
#     bar = fig.colorbar(im,
#                 ax=ax, orientation='vertical')
#     ticks = np.linspace(0, 1, 11)
#     bar.set_ticks(ticks=ticks, labels=map(lambda x: f'{x:.0%}', ticks))

# # https://en.wikipedia.org/wiki/Confusion_matrix#Confusion_matrices_in_multi-label_and_soft-label_classification
# def confusion_matrix(devices_flat, matches, metric='mean', w=180):
#     # convert the matches to a n x n matrix
#     fig, ax = plt.subplots()
#     n = len(devices_flat)
#     matrix = np.zeros((n, n))
#     for i, device in enumerate(devices_flat):
#         windows = len(matches[device][metric])
#         for name, count in Counter(matches[device][metric]).items():
#             share = count / windows

#             matrix[n - 1 - i][devices_flat.index(name)] = share


#     scores = []

#     for i in range(n):
#         n = len(matrix)
#         tp = matrix[n - 1 -i, i]
#         fp = 0
#         fn = 0
#         for j in range(n):
#             if j != i:
#                 fp += matrix[n - 1 - j][i]
#                 fn += matrix[n - 1 - i][j]
#         scores.append(2 * tp / (2 * tp + fp + fn))

#     my_colormesh(fig, ax, devices_flat[::-1] , [f'{device} {devices_to_cat[device]}' for i, device in enumerate(devices_flat)], matrix)

#     for i in range(n):
#         for j in range(n):
#             share = matrix[n - i - 1][j]
#             if share != 0:
#                 colour = 'black'
#                 if share > 0.1:
#                     colour = 'white'
#                 ax.text(n - i - 1, j, f'{share:.0%}', fontsize=3, ha='center',va='center', color=colour)
#     # TODO are FPs scaled correcly
#     # TODO there's an off by one error

#     ax.tick_params('x', rotation=90)
#     ax.set_ylabel("Real Class")
#     ax.set_xlabel("Predicted Class")
#     ax.set_title(f"Confusion Matrix:\n (state=idle, metric=mean, w = {w} s = 10) classified using median")
#     # fig.set_figwidth(50)
#     fig.set_dpi(200)

#     return matrix


# """
# Find overlapping intervals, return the number of non-overlapping intervals and the total amount of overlap.

# A faster algorithm would place all on same numberline, then it's like the brackets problem. n log n
# Do naive n^2 algo for now.

# For 2d overlaps, do both? Any overlap counts.

# >>> overlap_intervals((pd.Interval(left=1, right=3), pd.Interval(left=2, right=4), pd.Interval(left=10, right=20)))
# 1
# """
# def overlap_intervals(intervals, metrics=['mean']):
#     # TODO: look at multiple metrics here
#     for i in range(len(metrics)):
#         pass
#     # compare the two metrics for the same thing

#     independents = 0
#     for i, interval in enumerate(intervals):
#         independent = True
#         for j, compare in enumerate(intervals):
#             if i != j:
#                 if interval.overlaps(compare):
#                     independent = False
#         if independent:
#             independents += 1

#     return independents

doctest.testmod(verbose=False)

TestResults(failed=0, attempted=11)

In [20]:
c = Context('../PowerIDS/classifier/power_csv')

exp = Experiment(c, 180, 10, State.IDLE)
apply_windows(exp)

FileNotFoundError: [Errno 2] No such file or directory: '../PowerIDS/classifier/power_csv/appliance/<__main__.Device object at 0x76bee45f1af0>/active.csv'